In [1]:
# =========================================================
# One-Cell RAG (No LangChain/Chroma) — 安裝 + 程式一次搞定
# =========================================================

# ---------- 安裝必要套件 ----------
!pip -q install "sentence-transformers==3.0.1" "transformers==4.46.3" "huggingface_hub==0.34.1" \
                "rank-bm25==0.2.2" "jieba==0.42.1" "pillow==10.3.0" \
                "pytesseract==0.3.13" "pyyaml==6.0.2" "scikit-learn==1.5.2" \
                "openai==1.58.1"

# 安裝 Tesseract（含繁中語料）
!apt-get -y update >/dev/null
!apt-get -y install tesseract-ocr tesseract-ocr-chi-tra >/dev/null

# ---------- 限制平行化/執行緒，提升穩定性 ----------
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# ---------- 匯入 ----------
import re, glob, json, time, shutil
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple

import numpy as np
import yaml
import jieba
from PIL import Image
import pytesseract
from rank_bm25 import BM25Okapi

# 生成（可選；沒設 API Key 也能跑檢索）
from openai import OpenAI
import getpass

# ---------- 參數 ----------
OCR_ENABLED = False            # True：對資料夾內 .png 進行 OCR
EMBED_BATCH = 16              # e5-base 在 CPU 也可；避免過大 batch
K_VEC = 30                    # 向量檢索 top-k
K_BM25 = 50                   # BM25 檢索 top-k
TOP_N_FINAL = 10               # 最終合併後保留的段數（送 LLM 前可再做 small-to-big）

BASIC_CHUNK   = 200
BASIC_OVERLAP = 100
HONG_CHUNK = 300
HONG_OVERLAP = 60

# ---------- Colab Drive（可用時自動掛載） ----------
try:
    from google.colab import drive as _colab_drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    try:
        _colab_drive.mount("/content/drive", force_remount=False)
    except Exception:
        _colab_drive.mount("/content/drive", force_remount=True)

BASE_DIR = "/content/drive/MyDrive/v4_museum_rag_data" if IN_COLAB else "./data"
Path(BASE_DIR).mkdir(parents=True, exist_ok=True)

# ---------- OpenAI（若沒設定則只做檢索） ----------
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4.1-mini")

client = None
USE_LLM = False

def init_openai():
    global client, USE_LLM
    key = os.environ.get("OPENAI_API_KEY", "").strip()
    if not key:
        try:
            key = getpass.getpass("請貼上你的 OpenAI API Key（可按 Enter 跳過）：").strip()
        except Exception:
            key = ""
        if key:
            os.environ["OPENAI_API_KEY"] = key

    key = os.environ.get("OPENAI_API_KEY", "").strip()
    if not key:
        print("未設定 OPENAI_API_KEY，系統將只做檢索，不進行回答生成。")
        USE_LLM = False
        return

    try:
        client = OpenAI(api_key=key)
        USE_LLM = True
        print(f"OpenAI 初始化成功，使用模型：{OPENAI_MODEL}")
    except Exception as e:
        print(f"OpenAI 初始化失敗：{e}，改為只做檢索。")
        USE_LLM = False

init_openai()

# ---------- 小工具 ----------
def zh_tokens(text: str) -> List[str]:
    return [t.strip() for t in jieba.lcut(text or "") if t.strip()]

def chunk_text(txt: str, size: int, overlap: int) -> List[str]:
    txt = (txt or "").strip()
    if not txt:
        return []
    out, i, n = [], 0, len(txt)
    step = max(1, size - overlap)
    while i < n:
        out.append(txt[i:i+size])
        i += step
    return out

@dataclass
class Document:
    page_content: str
    metadata: Dict[str, Any]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.8/558.8 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.3/454.3 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 27.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.9.post2 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
W: Sk

/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")


Mounted at /content/drive
請貼上你的 OpenAI API Key（可按 Enter 跳過）：··········
OpenAI 初始化成功，使用模型：gpt-4.1-mini


In [2]:
# ---------- Front-matter 解析（支援 BOM/YAML/fallback） ----------
FRONT_MATTER_RE = re.compile(r"^\ufeff?---\s*\n(.*?)\n---\s*\n?", re.DOTALL)

def parse_front_matter(text: str) -> dict:
    m = FRONT_MATTER_RE.match(text or "")
    if not m:
        return {}
    block = m.group(1)
    # 優先 YAML
    try:
        data = yaml.safe_load(block)
        if isinstance(data, dict):
            return {str(k).strip(): (str(v).strip() if isinstance(v, str) else v)
                    for k, v in data.items() if k is not None}
    except Exception:
        pass
    # fallback：key: value
    meta = {}
    for line in block.splitlines():
        line = line.strip()
        if not line or line.startswith("#") or ":" not in line:
            continue
        k, v = line.split(":", 1)
        k = k.strip(); v = v.strip()
        if not k:
            continue
        if (len(v) >= 2) and ((v[0] == v[-1]) and v[0] in ("'", '"')):
            v = v[1:-1].strip()
        meta[k] = v
    return meta

def strip_front_matter(text: str) -> str:
    return FRONT_MATTER_RE.sub("", text, count=1) if FRONT_MATTER_RE.match(text or "") else (text or "")

def infer_domain_from_name(path: str) -> str:
    name = Path(path).name.lower()
    if name.endswith("_basic.md"):
        return "basic"
    if name.endswith("_hongloumeng.md"):
        return "hongloumeng"
    return "basic"

# --- Pillow ↔ transformers 熱修：補上 PIL._util.is_directory（若不存在）---
try:
    import PIL
    import PIL._util as _pil_util
    if not hasattr(_pil_util, "is_directory"):
        import os as _os
        def _is_directory(path):
            try:
                return _os.path.isdir(path)
            except Exception:
                return False
        _pil_util.is_directory = _is_directory
except Exception:
    pass

# ---------- 載入 Markdown ----------
def load_md_as_docs_by_domain(base_dir: str) -> Tuple[List[Document], List[Document]]:
    docs_basic, docs_hong = [], []
    md_files = glob.glob(str(Path(base_dir) / "**/*.md"), recursive=True) \
             + glob.glob(str(Path(base_dir) / "**/*.markdown"), recursive=True)
    for p in md_files:
        raw = ""
        try:
            raw = Path(p).read_text(encoding="utf-8", errors="strict")
        except Exception:
            try:
                raw = Path(p).read_text(encoding="utf-8", errors="ignore")
            except Exception:
                raw = Path(p).read_text(encoding="latin-1", errors="ignore")
        meta = parse_front_matter(raw)
        body = strip_front_matter(raw)
        domain = (meta.get("domain") or infer_domain_from_name(p) or "basic").strip()
        stem = Path(p).stem
        obj = (meta.get("object_name") or (stem.split("_")[0] if "_" in stem else stem) or "unknown").strip()
        doc = Document(
            page_content=body,
            metadata={
                "source": str(p),
                "object_name": obj,
                "domain": domain,
                "type": "markdown"
            }
        )
        (docs_hong if domain == "hongloumeng" else docs_basic).append(doc)
    return docs_basic, docs_hong

docs_basic, docs_hong = load_md_as_docs_by_domain(BASE_DIR)
print(f"[LOAD] basic={len(docs_basic)} hongloumeng={len(docs_hong)}")

# ---------- PNG OCR（可選） ----------
def load_png_ocr_as_docs(base_dir: str) -> List[Document]:
    out = []
    pngs = glob.glob(str(Path(base_dir) / "**/*.png"), recursive=True)
    for p in pngs:
        try:
            txt = pytesseract.image_to_string(Image.open(p), lang="chi_tra").strip()
            if txt:
                out.append(Document(
                    page_content=txt,
                    metadata={"source": str(p), "object_name": Path(p).stem, "domain": "basic", "type": "png_ocr"}
                ))
        except Exception as e:
            print(f"[WARN] OCR 失敗：{p} -> {e}")
    return out

if OCR_ENABLED:
    png_docs = load_png_ocr_as_docs(BASE_DIR)
    docs_basic += png_docs
    print(f"[LOAD] 加入 OCR -> basic={len(docs_basic)}")
else:
    print("[LOAD] 已跳過 PNG OCR（OCR_ENABLED=False）")

[LOAD] basic=34 hongloumeng=9
[LOAD] 已跳過 PNG OCR（OCR_ENABLED=False）


## 分段（加入 span 以便 IoU）

In [3]:
# ---------- 分段（加入 span 以便 IoU） ----------
def split_docs(docs: List[Document], size: int, overlap: int, add_parent: bool=False) -> List[Document]:
    out = []
    step = max(1, size - overlap)
    for d in docs:
        text = (d.page_content or "")
        n, i, chunk_idx = len(text), 0, 0
        while i < n:
            j = min(i + size, n)
            ch = text[i:j]
            meta = dict(d.metadata)
            if add_parent:
                meta["parent_id"] = meta.get("source")
            meta["span_start"] = i
            meta["span_end"]   = j
            meta["chunk_idx"]  = chunk_idx
            out.append(Document(page_content=ch, metadata=meta))
            i += step
            chunk_idx += 1
    return out

splits_basic = split_docs(docs_basic, BASIC_CHUNK, BASIC_OVERLAP, add_parent=False)
splits_hong  = split_docs(docs_hong , HONG_CHUNK , HONG_OVERLAP , add_parent=True)
print(f"[SPLIT] basic_chunks={len(splits_basic)} hong_chunks={len(splits_hong)}")


[SPLIT] basic_chunks=251 hong_chunks=98


## 本地 e5-base 向量

In [4]:
# ---------- 本地 e5-base 嵌入 ----------
from sentence_transformers import SentenceTransformer

def build_local_e5(device=None, batch_size=EMBED_BATCH):
    import torch
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[E5] device={device}, batch_size={batch_size}")
    model = SentenceTransformer("intfloat/multilingual-e5-base", device=device)

    def embed_passages(texts: List[str]) -> np.ndarray:
        texts_p = [("passage: " + (t or "")).strip() for t in texts]
        vecs = model.encode(
            texts_p,
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        ).astype("float32")
        return vecs

    def embed_query(q: str) -> np.ndarray:
        q = ("query: " + (q or "")).strip()
        v = model.encode(
            [q],
            batch_size=1,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )[0].astype("float32")
        return v

    return embed_passages, embed_query

embed_passages, embed_query = build_local_e5()

def embed_docs(docs: List[Document]) -> np.ndarray:
    texts = [d.page_content for d in docs]
    if not texts:
        return np.zeros((0, 768), dtype="float32")
    vecs = embed_passages(texts)
    vecs = vecs / (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-8)  # 再保險正規化
    return vecs

print("[EMB] 開始建立 basic 向量…")
vecs_basic = embed_docs(splits_basic)
print(f"[EMB] basic 完成，shape={vecs_basic.shape}")

print("[EMB] 開始建立 hongloumeng 向量…")
vecs_hong = embed_docs(splits_hong)
print(f"[EMB] hongloumeng 完成，shape={vecs_hong.shape}")


/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


[E5] device=cpu, batch_size=16


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

[EMB] 開始建立 basic 向量…
[EMB] basic 完成，shape=(251, 768)
[EMB] 開始建立 hongloumeng 向量…
[EMB] hongloumeng 完成，shape=(98, 768)


## 檢索器（向量 / BM25）

In [5]:
# ---------- 檢索器 ----------
class NumpyVectorRetriever:
    """Cosine 相似度（mat 和 q 都是 L2-normalized → 內積=cosine）"""
    def __init__(self, docs: List[Document], mat: np.ndarray, k: int = K_VEC):
        self.docs = docs
        self.mat = (mat / (np.linalg.norm(mat, axis=1, keepdims=True) + 1e-8)) if len(mat) else mat
        self.k = k

    def search(self, query: str) -> List[Document]:
        if len(self.mat) == 0:
            return []
        q = embed_query(query)
        q = q / (np.linalg.norm(q) + 1e-8)
        sims = self.mat @ q
        idxs = np.argsort(-sims)[:self.k]
        return [self.docs[i] for i in idxs]

vec_basic_ret = NumpyVectorRetriever(splits_basic, vecs_basic, k=K_VEC)
vec_hong_ret  = NumpyVectorRetriever(splits_hong , vecs_hong , k=K_VEC)


class BM25Retriever:
    def __init__(self, docs: List[Document], k: int = K_BM25):
        self.docs = docs
        tokenized = [zh_tokens(d.page_content) for d in docs]
        self.k = k
        self.bm25 = BM25Okapi(tokenized)

    def search(self, query: str) -> List[Document]:
        toks = zh_tokens(query)
        if not toks:
            return []
        scores = self.bm25.get_scores(toks)
        order = np.argsort(-scores)[:self.k]
        return [self.docs[i] for i in order]

bm25_basic = BM25Retriever(splits_basic, k=K_BM25)
bm25_hong  = BM25Retriever(splits_hong , k=K_BM25)


Building prefix dict from the default dictionary ...
DEBUG:jieba:Building prefix dict from the default dictionary ...
Dumping model to file cache /tmp/jieba.cache
DEBUG:jieba:Dumping model to file cache /tmp/jieba.cache
Loading model cost 1.028 seconds.
DEBUG:jieba:Loading model cost 1.028 seconds.
Prefix dict has been built successfully.
DEBUG:jieba:Prefix dict has been built successfully.


## 合併去重（雜湊 key）

In [6]:
# ---------- 合併檢索 + 去重（以雜湊縮小 key 記憶體） ----------
import hashlib, json

def _doc_hash_key(d: Document) -> str:
    payload = (d.page_content or "") + "\n" + json.dumps(d.metadata or {}, sort_keys=True, ensure_ascii=False)
    return hashlib.md5(payload.encode("utf-8")).hexdigest()

def merge_dedup(dlists: List[List[Document]], limit: int) -> List[Document]:
    out, seen = [], set()
    for lst in dlists:
        for d in lst:
            key = _doc_hash_key(d)
            if key in seen:
                continue
            seen.add(key)
            out.append(d)
            if len(out) >= limit:
                return out
    return out[:limit]
# ----------（可選）TF-IDF 重排（支援中文 tokenizer） ----------
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

def tfidf_rerank(docs: List[Document], query: str, top_n: int = TOP_N_FINAL) -> List[Document]:
    if not docs:
        return []
    texts = [d.page_content for d in docs]
    vectorizer = TfidfVectorizer(tokenizer=zh_tokens, max_features=20000)
    X = vectorizer.fit_transform(texts + [query])
    sims = linear_kernel(X[-1], X[:-1]).flatten()
    order = np.argsort(-sims)[:top_n]
    return [docs[i] for i in order]
# ========= 調試輸出 & 互動確認 =========
def _short(txt: str, n: int = 200) -> str:
    txt = (txt or "").replace("\n", " ").replace("\r", " ")
    return txt[:n] + ("..." if len(txt) > n else "")

def _src_name(d: Document) -> str:
    return Path(d.metadata.get("source", "N/A")).name

def _print_list(title: str, docs: List[Document], limit_print: int = 12):
    print(f"{title}共 {len(docs)} 段：")
    for i, d in enumerate(docs[:limit_print], 1):
        print(f"[{i}] {_short(d.page_content)} 〈來源:{_src_name(d)}〉")

def _suggest_objects(candidates: List[Document], max_n: int = 5) -> List[str]:
    # 依 object_name 出現次數排序；並保留首次出現順序作次序 tie-break
    first_idx = {}
    freq: Dict[str, int] = {}
    for idx, d in enumerate(candidates):
        obj = (d.metadata.get("object_name") or "").strip()
        if not obj:
            continue
        freq[obj] = freq.get(obj, 0) + 1
        if obj not in first_idx:
            first_idx[obj] = idx
    ranked = sorted(freq.items(), key=lambda kv: (-kv[1], first_idx[kv[0]]))
    return [name for name, _ in ranked[:max_n]]

def preview_retrieval(question: str, limit_print: int = 12) -> Dict[str, List[Document]]:
    """列出 BM25、混合（未重排）、重排後，並回傳各階段結果方便後續使用。"""
    # 1) 各域 BM25
    bm25_basic_docs = bm25_basic.search(question)
    bm25_hong_docs  = bm25_hong.search(question)
    bm25_all = (bm25_basic_docs + bm25_hong_docs)

    # 2) 各域向量檢索
    vec_basic_docs = vec_basic_ret.search(question)
    vec_hong_docs  = vec_hong_ret.search(question)

    # 3) 各域混合（未重排）
    cand_basic = merge_dedup([vec_basic_docs, bm25_basic_docs], limit=K_VEC)
    cand_hong  = merge_dedup([vec_hong_docs , bm25_hong_docs ], limit=K_VEC)

    # 4) 重排（各域）
    rerank_basic = tfidf_rerank(cand_basic, question, top_n=TOP_N_FINAL)
    rerank_hong  = tfidf_rerank(cand_hong , question, top_n=TOP_N_FINAL)

    # 5) 合併視角（未重排→重排）
    cand_all   = merge_dedup([cand_basic, cand_hong], limit=K_VEC * 2)
    rerank_all = tfidf_rerank(cand_all, question, top_n=TOP_N_FINAL)

    # ---- 輸出 ----
    print(f"🔎 問題： {question}\n")
    _print_list("🧩 BM25 檢索結果", bm25_all, limit_print); print()
    _print_list("🔍 混合檢索（未重排）結果", cand_all, limit_print); print()
    _print_list(f"📄 重排序後保留 {len(rerank_all)} 段（top_n={TOP_N_FINAL}）：", rerank_all, limit_print)

    return {
        "bm25_all": bm25_all,
        "cand_basic": cand_basic,
        "cand_hong": cand_hong,
        "cand_all": cand_all,
        "rerank_basic": rerank_basic,
        "rerank_hong": rerank_hong,
        "rerank_all": rerank_all,
    }

def _filter_by_object(docs: List[Document], object_name: str) -> List[Document]:
    object_name = (object_name or "").strip()
    if not object_name:
        return docs
    return [d for d in docs if (d.metadata.get("object_name") or "").strip() == object_name]

In [7]:
# ---------- small-to-big（章回擴展） ----------
_src_cache: Dict[str, str] = {}

def read_source(path: str) -> str:
    if path not in _src_cache:
        try:
            _src_cache[path] = Path(path).read_text(encoding="utf-8", errors="ignore")
        except Exception:
            _src_cache[path] = ""  # 檔案不存在或讀取失敗時保底
    return _src_cache[path]

def format_docs_with_source(docs_: List[Document]) -> str:
    out = []
    for i, d in enumerate(docs_, 1):
        name = d.metadata.get("source","N/A").split("/")[-1]
        text = (d.page_content or "").strip()
        if not text:
            continue
        out.append(f"[{i}] {text}\n〈來源:{name}〉")
    return "\n\n".join(out)

def expand_parent_context(docs: List[Document], max_chars=8000) -> str:
    if not docs:
        return ""
    domain = docs[0].metadata.get("domain")
    if domain != "hongloumeng":
        return format_docs_with_source(docs)
    seen, buf = set(), []
    for d in docs:
        pid = d.metadata.get("parent_id") or d.metadata.get("source")
        if not pid or pid in seen:
            continue
        seen.add(pid)
        full = strip_front_matter(read_source(pid))
        name = Path(pid).name
        if full:
            buf.append(f"{full[:max_chars]}\n〈來源:{name}〉")
    return "\n\n".join(buf)

# ---------- Prompt & 生成 ----------
few_shot_example = """
**[範例一：展品資料可回答的事實性問題]**
【問題】 金鑲東珠貓睛石嬪妃朝冠頂，上面的朝冠頂是怎麼製作的？
【回答】 您好，很高興為您介紹這件精美的〈金鑲東珠貓睛石嬪妃朝冠頂〉！這件文物主要運用了清代非常精巧的「纍絲工藝」製作而成。工匠們將黃金拉成細絲，再巧妙地掐捏、編結、焊接，才得以呈現出如此華麗的樣貌。
文物特色：
• 工藝核心：以金絲編結出蓮花瓣底座及兩隻栩栩如生的鳳鳥。
• 頂端裝飾：最上方鑲嵌著一顆棕褐色的貓睛石。
• 珍寶點綴：鳳鳥身上及之間，點綴了大小珍珠，其中包含十一顆珍貴的東珠。
這件冠頂不僅展現了清代高超的珠寶工藝，也反映了當時嚴謹的服飾制度與等級規範。希望這些資訊能幫助您更了解這件文物！

**[範例二：展品主題相關的推論性問題]**
【問題】 金鑲東珠貓睛石嬪妃朝冠頂，為什麼妃子的冠頂要有兩隻鳳？
【回答】 您好，很高興為您介紹這件〈金鑲東珠貓睛石嬪妃朝冠頂〉！這件清代妃子佩戴的朝冠頂，以精巧纍絲工藝製成，其上裝飾著上下兩隻鳳鳥。鳳鳥作為后妃服飾的主要紋樣，象徵母儀天下與尊貴優雅的身分地位。
文物特色：
• 造型：冠頂設計有上下兩隻鳳鳥。
• 工藝：運用精湛纍絲技法製作。
• 材質：鑲嵌珍貴東珠與貓睛石。
然而，檢索內容中並未詳細說明妃子朝冠頂為何特別設計為「兩隻鳳鳥」的具體原因。若您對其他部分有興趣，歡迎隨時詢問！

**[範例三：完全無關問題]**
【問題】 金鑲東珠貓睛石嬪妃朝冠頂，製作者是誰？
【回答】 無相關資料。
""".strip()


PROMPT_TMPL = f"""
你是一位專業的文物導覽小幫手，專門幫助使用者了解、研究與欣賞歷史文物。
請以自然、親切、具故事感的「博物館導覽員」口吻回答。
【規則】
- 總字數控制在 100–150 字。
- 只依據【檢索內容】回答；即使 few-shot 範例中含具體事實，也不可引用，若檢索無資料請回覆「無相關資料」。
- 先用 2–4 句「說明」做連貫敘述；再條列 1–3 點補充，且不得與說明重複。

{few_shot_example}

【使用者問題】
{{question}}

【檢索內容】
{{context}}
""".strip()

# ---------- OpenAI LLM 初始化（替代 Gemini） ----------
USE_LLM = False
OPENAI_CLIENT = None
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4.1-mini")  # 這裡可以改成你要的模型名稱

def init_openai():
    """
    初始化 OpenAI，用於 RAG 回答生成。
    若沒設定 API key，就只跑檢索、不生成回答。
    """
    global USE_LLM, OPENAI_CLIENT

    key = os.environ.get("OPENAI_API_KEY", "").strip()
    if not key:
        try:
            key = getpass.getpass("請貼上你的 OpenAI API Key（可按 Enter 跳過，只用檢索）：").strip()
        except Exception:
            key = ""
        if key:
            os.environ["OPENAI_API_KEY"] = key

    key = os.environ.get("OPENAI_API_KEY", "").strip()
    if not key:
        USE_LLM = False
        print("⚠️ OPENAI_API_KEY 未設定，改為『只檢索、不生成』模式。")
        return

    try:
        OPENAI_CLIENT = OpenAI(api_key=key)
        USE_LLM = True
        print(f"✅ OpenAI 啟用成功，模型：{OPENAI_MODEL}")
    except Exception as e:
        USE_LLM = False
        print(f"⚠️ OpenAI 初始化失敗：{e}；改為『只檢索、不生成』模式。")

init_openai()

def llm_generate(prompt: str) -> str:
    """
    統一的 LLM 呼叫介面：現在改用 OpenAI。
    其他程式（ask / run_batch_rag）不需要改。
    """
    if not USE_LLM or OPENAI_CLIENT is None:
        return ""

    try:
        resp = OPENAI_CLIENT.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
        )
        return resp.choices[0].message.content or ""
    except Exception as e:
        print(f"[ERROR] LLM generation failed: {e}")
        return f"Error generating response: {e}"

# —— 你需要一個 llm_generate() 來發送 prompt ——
def llm_generate(prompt: str) -> str:
    """用 OpenAI 生成回答，給 ask()/run_batch_rag 用"""
    if not USE_LLM or client is None:
        return ""  # 沒有 LLM 就回空字串，外面只會看到檢索內容

    try:
        resp = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": "你是一位專業的博物館導覽員，請用自然的繁體中文回答。"},
                {"role": "user", "content": prompt},
            ],
            temperature=0.3,
        )
        return resp.choices[0].message.content
    except Exception as e:
        print(f"[ERROR] LLM generation failed: {e}")
        return f"Error generating response: {e}"

# ---------- 查詢流程 ----------
def format_docs_with_source(docs_: List[Document]) -> str:
    out = []
    for i, d in enumerate(docs_, 1):
        name = d.metadata.get("source","N/A").split("/")[-1]
        text = (d.page_content or "").strip()
        if not text:
            continue
        out.append(f"[{i}] {text}\n〈來源:{name}〉")
    return "\n\n".join(out)

def build_context_for_prompt(hits_basic: List[Document], hits_hong: List[Document],
                             max_basic_chars=2500, max_hong_chars=8000) -> str:
    basic_text = format_docs_with_source(hits_basic)
    if len(basic_text) > max_basic_chars:
        basic_text = basic_text[:max_basic_chars]
    hong_text = expand_parent_context(hits_hong, max_chars=max_hong_chars)
    blocks = []
    if basic_text.strip():
        blocks.append("【basic】\n" + basic_text)
    if hong_text.strip():
        blocks.append("【hongloumeng】\n" + hong_text)
    return "\n\n".join(blocks)

def retrieve_two_domains(q: str) -> Tuple[List[Document], List[Document]]:
    # 各自：向量 + BM25 → 合併去重 → TF-IDF 重排（取 TOP_N_FINAL）
    cand_basic = merge_dedup([vec_basic_ret.search(q), bm25_basic.search(q)], limit=K_VEC)
    cand_hong  = merge_dedup([vec_hong_ret.search(q) , bm25_hong.search(q) ], limit=K_VEC)
    hits_basic = tfidf_rerank(cand_basic, q, top_n=TOP_N_FINAL)
    hits_hong  = tfidf_rerank(cand_hong , q, top_n=TOP_N_FINAL)
    # 物件名對齊（若 basic 命中某 object_name，優先挑相同 object 的 hong）
    if hits_basic and hits_hong:
        objs = {d.metadata.get("object_name") for d in hits_basic}
        aligned = [d for d in hits_hong if d.metadata.get("object_name") in objs]
        if aligned:
            hits_hong = aligned[:TOP_N_FINAL]
    return hits_basic, hits_hong

def ask(question: str, show_debug: bool = True):
  if show_debug:
    print("🔎 問題：", question)
  hb, hh = retrieve_two_domains(q=question) # pass question as q
  if show_debug:
    print(f"🧩 basic 命中 {len(hb)} 段；hong 命中 {len(hh)} 段")
  context = build_context_for_prompt(hb, hh)
  prompt = PROMPT_TMPL.format(question=question, context=context)
  if USE_LLM:
    ans = llm_generate(prompt)
    print("\n=== 生成回答 ===\n")
    print(ans if ans else "（模型未回傳文字）")
  else:
    print("\n=== 無 LLM 模式（僅顯示檢索內容片段） ===\n")
  print(context[:2000] + ("..." if len(context) > 2000 else ""))

print("✅ 初始化完成。使用範例：ask('金鑲東珠貓睛石嬪妃朝冠頂有什麼用途？')")

✅ OpenAI 啟用成功，模型：gpt-4.1-mini
✅ 初始化完成。使用範例：ask('金鑲東珠貓睛石嬪妃朝冠頂有什麼用途？')


In [8]:
def build_prompt(question: str, context: str) -> str:
    # 防呆：確保樣板存在
    try:
        tmpl = PROMPT_TMPL
    except NameError:
        raise RuntimeError("PROMPT_TMPL 尚未定義或尚未執行到。請先執行定義 PROMPT_TMPL 的 cell。")
    return tmpl.format(question=question, context=context)

def ask_with_confirmation(question: str, suggest_top: int = 5, limit_print: int = 12, use_llm: bool = True):
    print("[ask_with_confirmation] v2 with prompt-builder")  # 用來確認新版本生效

    # 1) 檢索預覽（BM25 / 混合 / 重排）
    bundle = preview_retrieval(question, limit_print=limit_print)

    # 2) 文物 disambiguation 候選（來自 cand_all）
    options = _suggest_objects(bundle["cand_all"], max_n=suggest_top)
    chosen = None
    if options:
        print("\n❓請問您想詢問的是下列哪一件文物？（輸入序號或完整名稱；直接 Enter 略過）")
        for i, name in enumerate(options, 1):
            print(f"{i}. {name}")
        user = input("您的選擇：").strip()
        if user.isdigit():
            i = int(user)
            if 1 <= i <= len(options):
                chosen = options[i - 1]
        elif user:
            norm_user = user.replace(" ", "").lower()
            for name in options:
                if name.replace(" ", "").lower() == norm_user:
                    chosen = name
                    break
            if chosen is None:
                chosen = user

    if chosen:
        print(f"\n✅ 已選擇文物：{chosen}")
    else:
        print("\n⚠️ 未選擇特定文物，將使用重排後的跨域結果。")

    # 3) 依選擇過濾→組裝 context
    hits_basic = bundle["rerank_basic"]
    hits_hong  = bundle["rerank_hong"]
    if chosen:
        hits_basic = _filter_by_object(hits_basic, chosen)
        hits_hong  = _filter_by_object(hits_hong , chosen)
        if not hits_basic and not hits_hong:
            hits_basic = bundle["rerank_basic"]
            hits_hong  = bundle["rerank_hong"]

    context = build_context_for_prompt(hits_basic, hits_hong)
    print("\n=== 所有檢索片段（送入 LLM 前全文） ===\n")
    print(context[:4000] + ("..." if len(context) > 4000 else ""))

    # 4) 生成回答
    if USE_LLM and use_llm:
        prompt = build_prompt(question, context)   # ← 關鍵：組 prompt
        ans = llm_generate(prompt) or "（模型未回傳文字）"
        print("\n=== 生成回答 ===\n")
        print(ans)
    else:
        print("\n（未啟用 LLM：僅展示檢索片段，不進行回答生成）")

In [9]:
# ================================================
# 1. 把 Document list 轉成 JSON list（給評估用）
# ================================================
import os, json, time
import pandas as pd
from pathlib import Path

def _docs_to_json_list(docs, max_len_each=1200):
    """把 Document 列表轉成可落檔的 list[dict]（給後續評估使用）"""
    out = []
    for d in docs:
        if not (d and getattr(d, "page_content", None)):
            continue
        text = (d.page_content or "").strip()
        if not text:
            continue
        if len(text) > max_len_each:
            text = text[:max_len_each] + "…"
        m = d.metadata or {}
        out.append({
            "text": text,
            "source": m.get("source", ""),
            "object_name": m.get("object_name", ""),
            "domain": m.get("domain", ""),
            "span_start": m.get("span_start", None),
            "span_end": m.get("span_end", None),
            "chunk_idx": m.get("chunk_idx", None),
        })
    return out

# ================================================
# 2. 定義：批次版 RAG → 輸出 generated_results.csv
# ================================================
def run_batch_rag(questions,
                  out_csv="./eval_data/generated/generated_results.csv",
                  limit=50,
                  max_basic_chars=2500,
                  max_hong_chars=8000,
                  sleep_sec=0.0):
    """
    批次把問題丟進 RAG，存出 generated_results.csv

    需要你前面已經定義：
    - retrieve_two_domains(question) -> (hits_basic, hits_hong)
    - build_context_for_prompt(hb, hh, ...)
    - build_prompt(question, context)
    - llm_generate(prompt)
    - USE_LLM (bool)
    """
    questions = list(questions)[:limit]
    rows, t0 = [], time.time()

    for i, q in enumerate(questions, 1):
        try:
            # 1) 檢索（兩域）→ 命中片段
            hb, hh = retrieve_two_domains(q)
            ctx_docs = _docs_to_json_list(hb) + _docs_to_json_list(hh)

            # 2) 組裝實際送 LLM 的文字 context
            ctx_text = build_context_for_prompt(
                hb, hh,
                max_basic_chars=max_basic_chars,
                max_hong_chars=max_hong_chars
            )

            # 3) 生成回答
            if USE_LLM:
                prompt = build_prompt(q, ctx_text)
                resp = llm_generate(prompt) or ""
            else:
                resp = ""

            rows.append({
                "qid": i,
                "query": q,
                "response": resp,
                "contexts": json.dumps(ctx_docs, ensure_ascii=False),
                "num_contexts": len(ctx_docs),
                "used_llm": bool(USE_LLM),
            })

            print(f"[{i}/{len(questions)}] OK 片段:{len(ctx_docs)} 回答字數:{len(resp)}")
            if sleep_sec > 0:
                time.sleep(sleep_sec)

        except Exception as e:
            print(f"[{i}/{len(questions)}] ERROR: {e}")
            rows.append({
                "qid": i,
                "query": q,
                "response": f"(error) {e}",
                "contexts": "[]",
                "num_contexts": 0,
                "used_llm": bool(USE_LLM),
            })

    out_path = Path(out_csv)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"\n✅ 批次完成：{len(rows)} 題，耗時 {time.time()-t0:.1f}s")
    print(f"📄 已輸出：{out_path}")

# ================================================
# 3. 從 questions.csv 讀取題目（Colab：上傳檔案）
# ================================================
from google.colab import files

print("請上傳 questions.csv（需包含欄位 'query'）")
uploaded = files.upload()   # ← 這裡選擇你的 questions.csv

# 取得上傳檔名（假設就是 questions.csv）
questions_name = next(iter(uploaded.keys()))
QUESTIONS_PATH = Path(questions_name)
print("✅ 已上傳：", QUESTIONS_PATH)

df_q = pd.read_csv(QUESTIONS_PATH, encoding="utf-8-sig")
assert "query" in df_q.columns, "questions.csv 需包含欄位 'query'"

qs = df_q["query"].dropna().astype(str).tolist()
print(f"👉 讀到 {len(qs)} 題，將送入批次（最多處理 50 題）")

# ================================================
# 4. 執行批次 RAG，輸出 generated_results.csv
# ================================================
out_csv = Path("./eval_data/generated/generated_results.csv")
out_csv.parent.mkdir(parents=True, exist_ok=True)

run_batch_rag(
    qs,
    out_csv=str(out_csv),
    limit=141   # 想多一點可以把 50 改掉
)

請上傳 questions.csv（需包含欄位 'query'）


Saving questions.csv to questions.csv
✅ 已上傳： questions.csv
👉 讀到 141 題，將送入批次（最多處理 50 題）


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


[1/141] OK 片段:14 回答字數:210
[2/141] OK 片段:19 回答字數:195
[3/141] OK 片段:12 回答字數:230
[4/141] OK 片段:11 回答字數:225
[5/141] OK 片段:11 回答字數:245
[6/141] OK 片段:17 回答字數:176
[7/141] OK 片段:14 回答字數:223
[8/141] OK 片段:19 回答字數:204
[9/141] OK 片段:20 回答字數:215
[10/141] OK 片段:12 回答字數:206
[11/141] OK 片段:16 回答字數:199
[12/141] OK 片段:20 回答字數:214
[13/141] OK 片段:20 回答字數:244
[14/141] OK 片段:18 回答字數:203
[15/141] OK 片段:16 回答字數:200
[16/141] OK 片段:13 回答字數:206
[17/141] OK 片段:19 回答字數:180
[18/141] OK 片段:19 回答字數:238
[19/141] OK 片段:12 回答字數:240
[20/141] OK 片段:13 回答字數:192
[21/141] OK 片段:14 回答字數:180
[22/141] OK 片段:17 回答字數:241
[23/141] OK 片段:18 回答字數:220
[24/141] OK 片段:15 回答字數:210
[25/141] OK 片段:16 回答字數:215
[26/141] OK 片段:16 回答字數:178
[27/141] OK 片段:17 回答字數:210
[28/141] OK 片段:14 回答字數:217
[29/141] OK 片段:13 回答字數:218
[30/141] OK 片段:11 回答字數:221
[31/141] OK 片段:12 回答字數:222
[32/141] OK 片段:13 回答字數:205
[33/141] OK 片段:18 回答字數:235
[34/141] OK 片段:16 回答字數:195
[35/141] OK 片段:16 回答字數:221
[36/141] OK 片段:13 回答字數:272
[37/141] OK 片段:15 回答字數:200
[38/141] O

In [10]:
from google.colab import files

# 指定輸出的檔案路徑
file_path = "eval_data/generated/generated_results.csv"

# 下載到本機
files.download(file_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ---------------- 使用方式（自動偵測＋降級處理） ----------------
import pandas as pd
from pathlib import Path

def load_questions_csv():
    # 1) 先試你提供的路徑（本機/伺服器適用）
    candidate = Path(r"C:\Users\瑞苓\Desktop\questions\questions.csv")
    if candidate.exists():
        print("✅ 使用本機檔案：", candidate)
        return pd.read_csv(candidate, encoding="utf-8-sig"), candidate

    # 2) 嘗試在 Colab 掛載 Google Drive
    try:
        from google.colab import drive  # 只有 Colab 才有
        drive.mount('/content/drive', force_remount=False)
        gdrive_candidate = Path("/content/drive/MyDrive/questions/questions.csv")
        if gdrive_candidate.exists():
            print("使用 Google Drive 檔案：", gdrive_candidate)
            return pd.read_csv(gdrive_candidate, encoding="utf-8-sig"), gdrive_candidate
    except Exception:
        pass  # 不是 Colab 或尚未掛載，往下走

    # 3) 最後降級：Colab 手動上傳
    try:
        from google.colab import files
        print("找不到檔案。請上傳 questions.csv …")
        uploaded = files.upload()  # 會跳上傳選擇視窗
        up_name = next(iter(uploaded.keys()))
        up_path = Path(up_name)
        print("已上傳：", up_path)
        return pd.read_csv(up_path, encoding="utf-8-sig"), up_path
    except Exception as e:
        raise FileNotFoundError("無法取得題庫 questions.csv，請確認路徑或改用上傳方式。") from e

# 讀取題目
df_q, src_path = load_questions_csv()
assert "query" in df_q.columns, f"CSV 需包含欄位 'query'（來源：{src_path}）"

# 嘗試從 questions.csv 裡找出問題類型欄位（Y/F/N，來自 CATEGORY_DEFS）
label_col = None
for c in ["query_type", "category", "suppose_answer"]:
    if c in df_q.columns:
        label_col = c
        break

if label_col:
    print(f"偵測到問題類別欄位：{label_col}（預期值：Y / F / N）")
    # 建立 query -> Y/F/N 對照表
    label_map = df_q.set_index("query")[label_col].astype(str).str.upper().to_dict()
else:
    label_map = {}
    print("questions.csv 中沒有問題類別欄位（query_type / category / suppose_answer），"
          "稍後無法自動產生 suppose_answer(Y/F/N) ，需之後手動補標。")

# 題目列表（全部）
qs = df_q["query"].dropna().astype(str).tolist()
total_n = len(qs)
print(f"讀到 {total_n} 題，將全部送入批次以增加樣本多樣性")

# 決定輸出位置：若有來源檔，預設輸出到同資料夾；否則落在目前工作目錄
out_csv = (src_path.parent / "generated_results.csv") if src_path else Path("./eval_data/generated/generated_results.csv")
out_csv.parent.mkdir(parents=True, exist_ok=True)

# 執行批次 RAG：這裡改成用全部題目（例如 141 題）
run_batch_rag(
    qs,
    out_csv=str(out_csv),
    limit=total_n   # 不再限制 50 題，而是把全部題目都跑完
)
print("已輸出：", out_csv)

# --------------------------------------------------------
# 依「題目原本的類別 Y/F/N」加入 suppose_answer 欄位
# --------------------------------------------------------
df_gen = pd.read_csv(out_csv, encoding="utf-8-sig")

if label_map:
    # 以 query 對應回原題目類別
    df_gen["suppose_answer"] = df_gen["query"].map(label_map).fillna("")
    df_gen["suppose_answer"] = df_gen["suppose_answer"].str.upper()

    # 簡單檢查是否只包含 Y/F/N
    invalid = sorted(set(df_gen["suppose_answer"]) - {"Y", "F", "N", ""})
    if invalid:
        print("發現非預期標籤：", invalid,
              "（請檢查 questions.csv 裡的問題類別欄位是否只含 Y/F/N）")

    df_gen.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("已在 generated_results.csv 中加入 suppose_answer (Y / F / N) 欄位，"
          "定義來自 CATEGORY_DEFS："
          "Y=事實性、F=推論/詮釋、N=無關問題")
else:
    print("因缺少問題類別欄位，generated_results.csv 中未加入 suppose_answer。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
找不到檔案。請上傳 questions.csv …


Saving questions.csv to questions (2).csv
已上傳： questions (2).csv
偵測到問題類別欄位：suppose_answer（預期值：Y / F / N）
讀到 141 題，將全部送入批次以增加樣本多樣性


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 19197.06ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2833.75ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 7183.73ms


[1/141] OK 片段:20 回答字數:160


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 4274.47ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2859.64ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2076.67ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 8493.99ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 9656.88ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1241.29ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 21434.61ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encodi

...

[ERROR] LLM generation failed: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 250
Please retry in 5.184587482s.
[137/141] OK 片段:17 回答字數:551


[ERROR] LLM generation failed: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 250
Please retry in 4.325516118s.
[138/141] OK 片段:19 回答字數:551


[ERROR] LLM generation failed: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 250
Please retry in 3.495925748s.
[139/141] OK 片段:20 回答字數:551


[ERROR] LLM generation failed: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 250
Please retry in 2.591409583s.
[140/141] OK 片段:20 回答字數:551


[ERROR] LLM generation failed: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 250
Please retry in 1.718614388s.
[141/141] OK 片段:19 回答字數:551

 批次完成：141 題，耗時 2223.0s
已輸出：generated_results.csv
已輸出： generated_results.csv
已在 generated_results.csv 中加入 suppose_answer (Y / F / N) 欄位，定義來自 CATEGORY_DEFS：Y=事實性、F=推論/詮釋、N=無關問題


In [ ]:
import os

src_path = "generated_results.csv"

try:
    # 只有在 Colab 環境才有這個套件
    from google.colab import files

    if os.path.exists(src_path):
        print("✅ 找到檔案，準備下載：", src_path)
        files.download(src_path)  # 這行會在瀏覽器觸發下載到你的電腦
    else:
        print("⚠️ 找不到檔案：", src_path)

except ImportError:
    # 如果不是在 Colab（例如本機 Jupyter），就只提示路徑
    print("⚠️ 偵測不到 google.colab，無法自動觸發下載。")
    print("請手動從目前工作目錄下載檔案，路徑：", os.path.abspath(src_path))

✅ 找到檔案，準備下載： generated_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =========================================
# Chunker 評測模組：grid 跑多組 chunk_size/overlap
# =========================================

# 測的 chunker 清單（size, overlap）
CHUNKER_GRID = [
    (2000,1000), (1500,750), (1000,500), (600,300), (400,200), (200,100),
    (2000,0), (1500,0), (1000,0), (800,0), (600,0), (400,0), (200,0)
]

# ---- 小工具（交集長度 / IoU）----
def _overlap_len(a: Tuple[int,int], b: Tuple[int,int]) -> int:
    # 區間皆為 [start, end) 半開區間
    return max(0, min(a[1], b[1]) - max(a[0], b[0]))

def _iou(a: Tuple[int,int], b: Tuple[int,int]) -> float:
    inter = _overlap_len(a, b)
    if inter == 0:
        return 0.0
    union = (max(a[1], b[1]) - min(a[0], b[0]))
    return float(inter) / float(max(1, union))

# ---- 尋找 gold 目標所在的「母文件」與金標跨度 ----
def _find_parent_doc(docs_domain: List[Document], object_name: str) -> Document:
    cands = [d for d in docs_domain if (d.metadata.get("object_name") or "") == object_name]
    assert cands, f"[EVAL] 找不到 object_name={object_name} 的母文件"
    # 若同名多檔，取第一個（通常只有一份）
    return cands[0]

def _find_spans(full_text: str, phrases: List[str]) -> List[Tuple[int,int]]:
    spans = []
    for ph in phrases:
        ph = (ph or "").strip()
        if not ph:
            continue
        pos = full_text.find(ph)
        if pos >= 0:
            spans.append((pos, pos + len(ph)))
        else:
            print(f"[WARN] 找不到金標片段（請從原檔逐字複製）：{ph}")
    return spans

# ---- 依 chunker 建索引（不污染全域 splits/vecs）----
def _build_index_for_chunker(docs_domain: List[Document], domain: str,
                             size: int, overlap: int):
    add_parent = (domain == "hongloumeng")
    splits = split_docs(docs_domain, size=size, overlap=overlap, add_parent=add_parent)
    vecs = embed_docs(splits)
    vec_ret = NumpyVectorRetriever(splits, vecs, k=K_VEC)
    bm25_ret = BM25Retriever(splits, k=K_BM25)
    return splits, vec_ret, bm25_ret

# ---- 以單一 domain 檢索（向量 + BM25 → 去重 → TF-IDF 重排）----
def _retrieve_domain(vec_ret, bm25_ret, query: str, topk: int) -> List[Document]:
    cand = merge_dedup([vec_ret.search(query), bm25_ret.search(query)], limit=K_VEC)
    return tfidf_rerank(cand, query, top_n=topk)

# ---- 單一 chunker 的評測 ----
def evaluate_chunker(question: str,
                     gold_object_name: str,
                     gold_domain: str,
                     gold_phrases: List[str],
                     size: int, overlap: int,
                     topk: int = 10) -> Dict[str, Any]:
    # 1) 取 domain corpus 與母文件
    docs_domain = docs_basic if gold_domain == "basic" else docs_hong
    parent_doc = _find_parent_doc(docs_domain, gold_object_name)
    full_text = parent_doc.page_content or ""
    parent_source = parent_doc.metadata.get("source")

    # 2) 金標跨度
    gold_spans = _find_spans(full_text, gold_phrases)
    assert gold_spans, f"[EVAL] 金標 phrases 沒找到對應跨度：{gold_phrases}"

    # 3) 建索引 & 檢索
    splits, vec_ret, bm25_ret = _build_index_for_chunker(docs_domain, gold_domain, size, overlap)
    hits = _retrieve_domain(vec_ret, bm25_ret, question, topk=topk)

    # 4) 僅針對「同一件 object 的片段」計算指標（避免被其他物件稀釋）
    hits_same = [d for d in hits
                 if (d.metadata.get("object_name") == gold_object_name) and
                    (d.metadata.get("source") == parent_source)]

    denom = max(1, len(hits_same))  # 防 0
    helpful = []                    # 命中金標的 chunk
    omega_sum = 0.0                 # 加權精確率的分子
    best_iou = 0.0

    for d in hits_same:
        cs, ce = int(d.metadata.get("span_start", 0)), int(d.metadata.get("span_end", 0))
        cl = max(1, ce - cs)
        chunk_span = (cs, ce)

        hit = False
        best_iou_this = 0.0
        for gs in gold_spans:
            inter = _overlap_len(chunk_span, gs)
            if inter > 0:
                hit = True
                omega_sum += inter / float(cl)       # 對覆蓋度更敏感
                best_iou_this = max(best_iou_this, _iou(chunk_span, gs))

        if hit:
            helpful.append(d)
            best_iou = max(best_iou, best_iou_this)

    # 5) 指標
    recall = 1.0 if helpful else 0.0                        # 是否有任一命中
    precision = (len(helpful) / float(denom)) if denom else 0.0
    precision_omega = (omega_sum / float(denom)) if denom else 0.0

    return {
        "chunker": f"{size}_{overlap}",
        "retrieve": int(topk),

        "recall_mean": float(recall),           # 單題 → mean=值, std=0
        "recall_std": 0.0,

        "precision_mean": float(precision),
        "precision_std": 0.0,

        "precision_omega_mean": float(precision_omega),
        "precision_omega_std": 0.0,

        "iou_mean": float(best_iou),            # 取 top-k 中最佳 IoU
        "iou_std": 0.0,

        "hits_same": len(hits_same)             # 診斷用途：top-k 內同物件片段數
    }

# ---- 跑整個 chunker grid ----
def run_chunker_grid(question: str,
                     gold_object_name: str,
                     gold_domain: str,
                     gold_phrases: List[str],
                     topk: int = 10,
                     chunkers: List[Tuple[int,int]] = CHUNKER_GRID) -> List[Dict[str, Any]]:

    rows = []
    t0 = time.time()
    print(f"[GRID] 問題：{question}")
    print(f"[GRID] gold=({gold_domain} / {gold_object_name})，topk={topk}")
    print(f"[GRID] 總共 {len(chunkers)} 組 chunker 參數…\n")

    for (size, overlap) in chunkers:
        t1 = time.time()
        r = evaluate_chunker(question, gold_object_name, gold_domain, gold_phrases,
                             size=size, overlap=overlap, topk=topk)
        dt = time.time() - t1
        print(f" - {r['chunker']:>9s}  |  recall={r['recall_mean']:.2f}  "
              f"prec={r['precision_mean']:.2f}  ω-prec={r['precision_omega_mean']:.2f}  "
              f"IoU={r['iou_mean']:.2f}  | hits_same={r['hits_same']:>2d}  | {dt:.2f}s")
        rows.append(r)

    # 排序：先看 recall，再看加權 precision，再看 IoU
    rows.sort(key=lambda x: (-x["recall_mean"],
                             -x["precision_omega_mean"],
                             -x["iou_mean"],
                             -x["precision_mean"]))

    print("\n=== 📈 排名（前 10）===")
    for i, r in enumerate(rows[:10], 1):
        print(f"[{i:>2}] {r['chunker']:>9s} | R={r['recall_mean']:.2f}  "
              f"P={r['precision_mean']:.2f}  ωP={r['precision_omega_mean']:.2f}  "
              f"IoU={r['iou_mean']:.2f}")

    print(f"\n[GRID] 完成，用時 {time.time()-t0:.2f}s")
    return rows

# ---- 範例執行（把金標句子從原檔逐字貼過來）----
# 你這題：「這種白色的梅瓶是用來裝什麼的呢？」
QUESTION      = "這種白色的梅瓶是用來裝什麼的呢？"
GOLD_OBJECT   = "白瓷劃花蓮紋梅瓶"
GOLD_DOMAIN   = "basic"
GOLD_PHRASES  = [
    "上部橢圓並排短蓮瓣紋；前後器身中腹各以折枝蓮花為主紋，四重蓮瓣積疊，蓮瓣內以篾劃紋暗示其花片飽滿，兩側則勾勒簡單的荷葉陪襯；"
    "下部以雙線重覆勾劃細長蓮瓣形。全器施罩牙白釉，器表有積釉流淌的「淚痕」現象，足底緣無釉。"
]

# 跑評測（請先把上面 GOLD_PHRASES 換成逐字金標）

rows = run_chunker_grid(QUESTION, GOLD_OBJECT, GOLD_DOMAIN, GOLD_PHRASES, topk=10)
# 存成 CSV 看起來更清楚
import pandas as pd
pd.DataFrame(rows).to_csv("chunker_eval.csv", index=False)


[GRID] 問題：這種白色的梅瓶是用來裝什麼的呢？
[GRID] gold=(basic / 白瓷劃花蓮紋梅瓶)，topk=10
[GRID] 總共 13 組 chunker 參數…

 - 2000_1000  |  recall=1.00  prec=1.00  ω-prec=0.21  IoU=0.21  | hits_same= 1  | 82.01s
 -  1500_750  |  recall=1.00  prec=1.00  ω-prec=0.21  IoU=0.21  | hits_same= 1  | 85.15s
 -  1000_500  |  recall=1.00  prec=1.00  ω-prec=0.21  IoU=0.21  | hits_same= 1  | 104.52s
 -   600_300  |  recall=1.00  prec=1.00  ω-prec=0.34  IoU=0.44  | hits_same= 2  | 149.64s
 -   400_200  |  recall=1.00  prec=0.67  ω-prec=0.20  IoU=0.35  | hits_same= 3  | 169.20s
 -   200_100  |  recall=1.00  prec=0.60  ω-prec=0.21  IoU=0.53  | hits_same= 5  | 169.83s
 -    2000_0  |  recall=1.00  prec=1.00  ω-prec=0.21  IoU=0.21  | hits_same= 1  | 64.65s
 -    1500_0  |  recall=1.00  prec=1.00  ω-prec=0.21  IoU=0.21  | hits_same= 1  | 65.93s
 -    1000_0  |  recall=1.00  prec=1.00  ω-prec=0.21  IoU=0.21  | hits_same= 1  | 72.95s
 -     800_0  |  recall=1.00  prec=1.00  ω-prec=0.21  IoU=0.21  | hits_same= 1  | 80.96s
 -     600_0

In [ ]:
# =========================================
# Chunker 評測模組：grid 跑多組 chunk_size/overlap（可直接貼上）
# 依賴：docs_basic/docs_hong, split_docs, embed_docs, NumpyVectorRetriever,
#       BM25Retriever, merge_dedup, tfidf_rerank, K_VEC, K_BM25
# =========================================

from typing import Tuple, List, Dict, Any
import time
import pandas as pd

# ------- 檢查必要相依（若缺少就提醒）-------
_required = [
    "docs_basic","docs_hong","split_docs","embed_docs","NumpyVectorRetriever",
    "BM25Retriever","merge_dedup","tfidf_rerank","K_VEC","K_BM25"
]
missing = [k for k in _required if k not in globals()]
assert not missing, f"請先執行你的 RAG 初始化程式碼，缺少: {missing}"

# ------- 測的 chunker 清單（size, overlap）-------
CHUNKER_GRID = [
    (2000,1000), (1500,750), (1000,500), (600,300), (400,200), (200,100),
    (2000,0), (1500,0), (1000,0), (800,0), (600,0), (400,0), (200,0)
]

# ---- 小工具（交集長度 / IoU）----
def _overlap_len(a: Tuple[int,int], b: Tuple[int,int]) -> int:
    # 區間皆為 [start, end) 半開區間
    return max(0, min(a[1], b[1]) - max(a[0], b[0]))

def _iou(a: Tuple[int,int], b: Tuple[int,int]) -> float:
    inter = _overlap_len(a, b)
    if inter == 0:
        return 0.0
    union = (max(a[1], b[1]) - min(a[0], b[0]))
    return float(inter) / float(max(1, union))

# ---- 尋找 gold 目標所在的「母文件」與金標跨度 ----
def _find_parent_doc(docs_domain: List["Document"], object_name: str) -> "Document":
    cands = [d for d in docs_domain if (d.metadata.get("object_name") or "") == object_name]
    assert cands, f"[EVAL] 找不到 object_name={object_name} 的母文件"
    return cands[0]  # 若同名多檔，取第一個（通常只有一份）

def _find_spans(full_text: str, phrases: List[str]) -> List[Tuple[int,int]]:
    spans = []
    for ph in phrases:
        ph = (ph or "").strip()
        if not ph:
            continue
        pos = full_text.find(ph)
        if pos >= 0:
            spans.append((pos, pos + len(ph)))
        else:
            print(f"[WARN] 找不到金標片段（請從原檔逐字複製）：{ph}")
    return spans

# ---- 依 chunker 建索引（不污染全域 splits/vecs）----
def _build_index_for_chunker(docs_domain: List["Document"], domain: str,
                             size: int, overlap: int):
    add_parent = (domain == "hongloumeng")
    splits = split_docs(docs_domain, size=size, overlap=overlap, add_parent=add_parent)
    vecs = embed_docs(splits)
    vec_ret = NumpyVectorRetriever(splits, vecs, k=K_VEC)
    bm25_ret = BM25Retriever(splits, k=K_BM25)
    return splits, vec_ret, bm25_ret

# ---- 以單一 domain 檢索（向量 + BM25 → 去重 → TF-IDF 重排）----
def _retrieve_domain(vec_ret, bm25_ret, query: str, topk: int) -> List["Document"]:
    cand = merge_dedup([vec_ret.search(query), bm25_ret.search(query)], limit=K_VEC)
    return tfidf_rerank(cand, query, top_n=topk)

# ---- 單一 chunker 的評測 ----
def evaluate_chunker(question: str,
                     gold_object_name: str,
                     gold_domain: str,
                     gold_phrases: List[str],
                     size: int, overlap: int,
                     topk: int = 10) -> Dict[str, Any]:
    # 1) 取 domain corpus 與母文件
    docs_domain = docs_basic if gold_domain == "basic" else docs_hong
    parent_doc = _find_parent_doc(docs_domain, gold_object_name)
    full_text = parent_doc.page_content or ""
    parent_source = parent_doc.metadata.get("source")

    # 2) 金標跨度
    gold_spans = _find_spans(full_text, gold_phrases)
    assert gold_spans, f"[EVAL] 金標 phrases 沒找到對應跨度：{gold_phrases}"

    # 3) 建索引 & 檢索
    splits, vec_ret, bm25_ret = _build_index_for_chunker(docs_domain, gold_domain, size, overlap)
    hits = _retrieve_domain(vec_ret, bm25_ret, question, topk=topk)

    # 4) 僅針對「同一件 object 的片段」計算指標（避免被其他物件稀釋）
    hits_same = [d for d in hits
                 if (d.metadata.get("object_name") == gold_object_name) and
                    (d.metadata.get("source") == parent_source)]

    denom = max(1, len(hits_same))  # 防 0
    helpful = []                    # 命中金標的 chunk
    omega_sum = 0.0                 # 加權精確率的分子
    best_iou = 0.0

    for d in hits_same:
        cs, ce = int(d.metadata.get("span_start", 0)), int(d.metadata.get("span_end", 0))
        cl = max(1, ce - cs)
        chunk_span = (cs, ce)

        hit = False
        best_iou_this = 0.0
        for gs in gold_spans:
            inter = _overlap_len(chunk_span, gs)
            if inter > 0:
                hit = True
                omega_sum += inter / float(cl)       # 對覆蓋度更敏感
                best_iou_this = max(best_iou_this, _iou(chunk_span, gs))

        if hit:
            helpful.append(d)
            best_iou = max(best_iou, best_iou_this)

    # 5) 指標
    recall = 1.0 if helpful else 0.0                         # 是否有任一命中
    precision = (len(helpful) / float(denom)) if denom else 0.0
    precision_omega = (omega_sum / float(denom)) if denom else 0.0

    return {
        "chunker": f"{size}_{overlap}",
        "retrieve": int(topk),

        "recall_mean": float(recall),           # 單題 → mean=值, std=0
        "recall_std": 0.0,

        "precision_mean": float(precision),
        "precision_std": 0.0,

        "precision_omega_mean": float(precision_omega),
        "precision_omega_std": 0.0,

        "iou_mean": float(best_iou),            # 取 top-k 中最佳 IoU
        "iou_std": 0.0,

        "hits_same": len(hits_same)             # 診斷用途：top-k 內同物件片段數
    }

# ---- 跑整個 chunker grid ----
def run_chunker_grid(question: str,
                     gold_object_name: str,
                     gold_domain: str,
                     gold_phrases: List[str],
                     topk: int = 10,
                     chunkers: List[Tuple[int,int]] = CHUNKER_GRID) -> List[Dict[str, Any]]:

    rows = []
    t0 = time.time()
    print(f"[GRID] 問題：{question}")
    print(f"[GRID] gold=({gold_domain} / {gold_object_name})，topk={topk}")
    print(f"[GRID] 總共 {len(chunkers)} 組 chunker 參數…\n")

    for (size, overlap) in chunkers:
        t1 = time.time()
        r = evaluate_chunker(question, gold_object_name, gold_domain, gold_phrases,
                             size=size, overlap=overlap, topk=topk)
        dt = time.time() - t1
        print(f" - {r['chunker']:>9s}  |  recall={r['recall_mean']:.2f}  "
              f"prec={r['precision_mean']:.2f}  ω-prec={r['precision_omega_mean']:.2f}  "
              f"IoU={r['iou_mean']:.2f}  | hits_same={r['hits_same']:>2d}  | {dt:.2f}s")
        rows.append(r)

    # 排序：先看 recall，再看加權 precision，再看 IoU
    rows.sort(key=lambda x: (-x["recall_mean"],
                             -x["precision_omega_mean"],
                             -x["iou_mean"],
                             -x["precision_mean"]))

    print("\n=== 📈 排名（前 10）===")
    for i, r in enumerate(rows[:10], 1):
        print(f"[{i:>2}] {r['chunker']:>9s} | R={r['recall_mean']:.2f}  "
              f"P={r['precision_mean']:.2f}  ωP={r['precision_omega_mean']:.2f}  "
              f"IoU={r['iou_mean']:.2f}")

    print(f"\n[GRID] 完成，用時 {time.time()-t0:.2f}s")
    return rows

# ---- 範例執行（把金標句子從原檔逐字貼過來）----
QUESTION      = "這種白色的梅瓶是用來裝什麼的呢？"
GOLD_OBJECT   = "白瓷劃花蓮紋梅瓶"
GOLD_DOMAIN   = "basic"
GOLD_PHRASES  = [
    "上部橢圓並排短蓮瓣紋；前後器身中腹各以折枝蓮花為主紋，四重蓮瓣積疊，蓮瓣內以篾劃紋暗示其花片飽滿，兩側則勾勒簡單的荷葉陪襯；",
    "下部以雙線重覆勾劃細長蓮瓣形。全器施罩牙白釉，器表有積釉流淌的「淚痕」現象，足底緣無釉。"
]

# 跑評測（可改 topk）
rows = run_chunker_grid(QUESTION, GOLD_OBJECT, GOLD_DOMAIN, GOLD_PHRASES, topk=10)

# 存成 CSV 看起來更清楚
df_chunk_eval = pd.DataFrame(rows)
df_chunk_eval.to_csv("chunker_eval.csv", index=False)
print("\n✅ 已輸出：chunker_eval.csv")
print("🏁 建議 baseline（依排序第一名）：", df_chunk_eval.iloc[0]["chunker"])

In [ ]:
# ============================
# 列出 13 組 chunk_size/overlap 的實際命中片段
# ============================

from dataclasses import asdict
import pandas as pd

# 評測的 13 組參數（名稱, size, overlap）
CHUNKERS = [
    ("2000_1000", 2000, 1000),
    ("1500_750",  1500,  750),
    ("1000_500",  1000,  500),
    ("600_300",     600,  300),
    ("400_200",     400,  200),
    ("200_100",     200,  100),
    ("2000_0",     2000,    0),
    ("1500_0",     1500,    0),
    ("1000_0",     1000,    0),
    ("800_0",        800,    0),
    ("600_0",        600,    0),
    ("400_0",        400,    0),
    ("200_0",        200,    0),
]

def _short(txt: str, n: int = 180) -> str:
    txt = (txt or "").replace("\n", " ").replace("\r", " ")
    return txt[:n] + ("..." if len(txt) > n else "")

def build_index_for_chunker(size: int, overlap: int):
    """用單一 size/overlap 同步切 basic/hong，回傳可檢索的索引物件們。"""
    sb = split_docs(docs_basic, size, overlap, add_parent=False)
    sh = split_docs(docs_hong , size, overlap, add_parent=True)
    vb = embed_docs(sb)
    vh = embed_docs(sh)
    idx = {
        "splits_basic": sb, "splits_hong": sh,
        "vec_basic": vb, "vec_hong": vh,
        "vec_ret_basic": NumpyVectorRetriever(sb, vb, k=K_VEC),
        "vec_ret_hong":  NumpyVectorRetriever(sh, vh, k=K_VEC),
        "bm25_basic":    BM25Retriever(sb, k=K_BM25),
        "bm25_hong":     BM25Retriever(sh, k=K_BM25),
    }
    return idx

def retrieve_for_index(question: str, index: dict, topk: int = 10):
    """用指定索引做 向量+BM25 混合 → 去重 → TFIDF 重排，回傳 basic/hong 的 topk 文件。"""
    cand_basic = merge_dedup([
        index["vec_ret_basic"].search(question),
        index["bm25_basic"].search(question)
    ], limit=K_VEC)
    cand_hong = merge_dedup([
        index["vec_ret_hong"].search(question),
        index["bm25_hong"].search(question)
    ], limit=K_VEC)

    hits_basic = tfidf_rerank(cand_basic, question, top_n=topk)
    hits_hong  = tfidf_rerank(cand_hong,  question, top_n=topk)

    # 讓 hong 與 basic 的 object 對齊（若對你有幫助）
    if hits_basic and hits_hong:
        objs = {d.metadata.get("object_name") for d in hits_basic}
        aligned = [d for d in hits_hong if d.metadata.get("object_name") in objs]
        if aligned:
            hits_hong = aligned[:topk]
    return hits_basic, hits_hong

def list_hits_all_chunkers(question: str, topk: int = 10, save_csv_path: str = None):
    rows = []
    print(f"[GRID-HITS] 問題：{question}\n")
    for name, size, ov in CHUNKERS:
        print(f"\n================ {name} (size={size}, overlap={ov}) ================\n")
        t0 = time.time()
        idx = build_index_for_chunker(size, ov)
        hb, hh = retrieve_for_index(question, idx, topk=topk)
        dt = time.time() - t0

        def _emit(domain_name: str, docs: List[Document]):
            for r, d in enumerate(docs, start=1):
                meta = d.metadata or {}
                rows.append({
                    "chunker": name,
                    "domain": domain_name,
                    "rank": r,
                    "object_name": meta.get("object_name", ""),
                    "source": Path(meta.get("source","")).name,
                    "chunk_idx": meta.get("chunk_idx", None),
                    "span_start": meta.get("span_start", None),
                    "span_end":   meta.get("span_end", None),
                    "char_len": len(d.page_content or ""),
                    "snippet": _short(d.page_content, 200),
                })

        # 印出 basic 命中
        print("【basic】")
        for i, d in enumerate(hb, 1):
            m = d.metadata or {}
            print(f"[B{i}] {m.get('object_name','')} | {Path(m.get('source','')).name} | "
                  f"span={m.get('span_start')}–{m.get('span_end')} | idx={m.get('chunk_idx')} | "
                  f"{_short(d.page_content, 160)}")
        _emit("basic", hb)

        # 印出 hong 命中
        print("\n【hongloumeng】")
        for i, d in enumerate(hh, 1):
            m = d.metadata or {}
            print(f"[H{i}] {m.get('object_name','')} | {Path(m.get('source','')).name} | "
                  f"span={m.get('span_start')}–{m.get('span_end')} | idx={m.get('chunk_idx')} | "
                  f"{_short(d.page_content, 160)}")
        _emit("hongloumeng", hh)

        print(f"\n⏱️ 用時：{dt:.2f}s")

    df = pd.DataFrame(rows, columns=[
        "chunker","domain","rank","object_name","source","chunk_idx","span_start","span_end","char_len","snippet"
    ])
    if save_csv_path:
        out_path = Path(save_csv_path)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out_path, index=False, encoding="utf-8-sig")
        print(f"\n[GRID-HITS] 已輸出 CSV：{out_path}（共 {len(df)} 筆）")
    return df

# 執行：把你的問題填進來（topk 建議用 10，和你前面評測一致）
QUESTION = "這種白色的梅瓶是用來裝什麼的呢？"
hits_df = list_hits_all_chunkers(
    question=QUESTION,
    topk=10,
    save_csv_path=f"{BASE_DIR}/grid_hits_meiping.csv"
)

# 前幾列
print("\n--- 首 20 筆 ---")
print(hits_df.head(20).to_string(index=False))

[GRID-HITS] 問題：這種白色的梅瓶是用來裝什麼的呢？


================ 2000_1000 (size=2000, overlap=1000) ================

【basic】
[B1] 白瓷劃花蓮紋梅瓶 | 白瓷劃花蓮紋梅瓶_basic.md | span=0–503 | idx=0 | # 白瓷劃花蓮紋梅瓶  ## 基本資訊 - **英文品名**: Meiping vase with incised lotus decoration in white glaze, Ding ware - **統一編號**: 故瓷013471N000000000 - **分類**: 陶瓷器 - **時代**: 北宋 西...
[B2] 銅鍍金嵌瑪瑙修妝匣 | 銅鍍金嵌瑪瑙修妝匣_basic.md | span=0–398 | idx=0 | # 銅鍍金嵌瑪瑙修妝匣  ## 基本資訊 - **英文品名**: Gilt bronze necessaire decorated with agate inlay - **統一編號**: 故雜000653N000000000 - **分類**: 雜項 - **時代**: 西元18世紀 - **尺寸**: 長6.1公分...
[B3] 銀鍍金鏤空芙蓉花指甲套 | 銀鍍金鏤空芙蓉花指甲套_basic.md | span=0–405 | idx=0 | # 銀鍍金鏤空芙蓉花指甲套  ## 基本資訊 - **英文品名**: Gilt silver openwork fingernail guard with cotton rose decoration - **統一編號**: 故雜003610N000000000 - **分類**: 雜項 - **時代**: 清（164...
[B4] 銅胎畫琺瑯黃地花卉紋套杯 | 銅胎畫琺瑯黃地花卉紋套杯_basic.md | span=0–511 | idx=0 | # 銅胎畫琺瑯黃地花卉紋套杯  ## 基本資訊 - **英文品名**: Gilt copper cups with painted enamel floral patterns on yellow grounds (set of six) - **統一編號**: 故琺000215N000000000 

KeyboardInterrupt: 